In [1]:

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
import ast
from scipy.sparse import hstack


In [2]:
df = pd.read_csv(r"C:\Users\ADITYA\Downloads\top_1000_popular_movies_tmdb.csv", engine='python', on_bad_lines='skip')
df.head()


,Unnamed: 0,id,title,release_date,genres,original_language,vote_average,vote_count,popularity,overview,budget,production_companies,revenue,runtime,tagline
0,0,385687,Fast X,2023-05-17,"['Action', 'Crime', 'Thriller']",English,7.4,1347.0,8363.473,Over many missions and against impossible odds...,340000000.0,"['Universal Pictures', 'Original Film', 'One R...",6.520000e+08,142.0,The end of the road begins.
1,1,603692,John Wick: Chapter 4,2023-03-22,"['Action', 'Thriller', 'Crime']",English,7.9,2896.0,4210.313,"With the price on his head ever increasing, Jo...",90000000.0,"['Thunder Road', '87Eleven', 'Summit Entertain...",4.317692e+08,170.0,"No way back, one way out."
2,2,502356,The Super Mario Bros. Movie,2023-04-05,"['Animation', 'Family', 'Adventure', 'Fantasy'...",English,7.8,4628.0,3394.458,"While working underground to fix a water main,...",100000000.0,"['Universal Pictures', 'Illumination', 'Ninten...",1.308767e+09,92.0,NaN
3,3,569094,Spider-Man: Across the Spider-Verse,2023-05-31,"['Action', 'Adventure', 'Animation', 'Science ...",English,8.8,1160.0,2859.047,"After reuniting with Gwen Stacy, Brooklyn’s fu...",100000000.0,"['Columbia Pictures', 'Sony Pictures Animation...",3.135222e+08,140.0,It's how you wear the mask that matters
4,4,536437,Hypnotic,2023-05-11,"['Mystery', 'Thriller', 'Science Fiction']",English,6.5,154.0,2654.854,A detective becomes entangled in a mystery inv...,70000000.0,"['Studio 8', 'Solstice Productions', 'Ingeniou...",0.000000e+00,94.0,Control is an illusion.


In [3]:
# Step 3: Data Cleaning
# Drop rows with missing 'overview' or 'genres'
df = df.dropna(subset=['overview', 'genres'])

# Convert genre strings to lists
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

df[['title', 'genres', 'overview']].head()


,title,genres,overview
0,Fast X,"[Action, Crime, Thriller]",Over many missions and against impossible odds...
1,John Wick: Chapter 4,"[Action, Thriller, Crime]","With the price on his head ever increasing, Jo..."
2,The Super Mario Bros. Movie,"[Animation, Family, Adventure, Fantasy, Comedy]","While working underground to fix a water main,..."
3,Spider-Man: Across the Spider-Verse,"[Action, Adventure, Animation, Science Fiction]","After reuniting with Gwen Stacy, Brooklyn’s fu..."
4,Hypnotic,"[Mystery, Thriller, Science Fiction]",A detective becomes entangled in a mystery inv...


In [4]:
# Step 4: Encode Genres with MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df['genres'])

genre_df = pd.DataFrame(genre_encoded, columns=mlb.classes_)
df = pd.concat([df, genre_df], axis=1)



In [5]:
# Fill NaN overviews with empty strings
df['overview'] = df['overview'].fillna('')

# TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'])


In [6]:
# Clean and reset index to ensure alignment
df = df.dropna(subset=['overview', 'genres']).reset_index(drop=True)

# Convert genres from string to list
import ast
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

# Fill any remaining NaNs in overview
df['overview'] = df['overview'].fillna('')

# Recreate genre_encoded and tfidf_matrix after cleaning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import hstack, csr_matrix

# TF-IDF on overview
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'])

# Encode genres
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df['genres'])

# Convert genre_encoded to sparse matrix
genre_sparse = csr_matrix(genre_encoded)

# Now safely stack them
combined_features = hstack([tfidf_matrix, genre_sparse])
combined_features.shape


(9924, 28068)

In [7]:
# Step 7: Compute Cosine Similarity Matrix
cosine_sim = cosine_similarity(combined_features, combined_features)


In [8]:
# Step 8: Define Recommendation Function
def recommend_movies(title, df=df, similarity=cosine_sim):
    title = title.lower()
    indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

    if title not in indices:
        return f"'{title}' not found in dataset."
    
    idx = indices[title]
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]  # Top 10 similar movies

    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]


In [21]:
# Step 9: Try an Example!
recommend_movies("Fast X")


6456                   Fast Five
4369              Fast & Furious
204      The Fate of the Furious
178                    Furious 7
9823    The Fast and the Furious
8275               Dom Hemingway
139                           F9
395                     Creed II
1096      Mechanic: Resurrection
6699     Coming Home in the Dark
Name: title, dtype: object